# 🧠 Alpha Connectome (Wunder RNN) — Exploratory Data Analysis

**Run each cell one at a time.** Every cell is small and fast.

> ⚠️ NEVER call `pd.read_parquet()` on the full files — train is **29 GB**, valid is **5.6 GB**.  
> Always use `ParquetFile.read_row_group(i)` to load **one sequence (20 000 rows)** at a time.

---
| File | Sequences | Rows |
|------|----------:|-----:|
| `train.parquet` | 10 607 | 212 140 000 |
| `valid.parquet` | 1 873 | 37 460 000 |
| `valid_mask.parquet` | 1 873 | 37 460 000 |


## Cell 1 — Imports & Paths

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if not (ROOT / 'utils.py').exists() and (ROOT.parent / 'utils.py').exists():
    ROOT = ROOT.parent
print('ROOT:', ROOT)

assert (ROOT / 'utils.py').exists(), 'utils.py not found — run from the pack root'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import FEATURE_COLUMNS, TARGET_COLUMNS, SEQUENCE_LENGTH, WARMUP

TRAIN_PATH = ROOT / 'datasets' / 'train.parquet'
VALID_PATH = ROOT / 'datasets' / 'valid.parquet'
MASK_PATH  = ROOT / 'datasets' / 'valid_mask.parquet'

print(f'Features : {len(FEATURE_COLUMNS)}, Targets: {TARGET_COLUMNS}')
print(f'Sequence length: {SEQUENCE_LENGTH:,}, Warm-up: {WARMUP} steps')
print('All paths exist:', all(p.exists() for p in [TRAIN_PATH, VALID_PATH, MASK_PATH]))


## Cell 2 — Parquet Metadata (reads headers only, no data)

In [ ]:
train_pf = pq.ParquetFile(TRAIN_PATH)
valid_pf = pq.ParquetFile(VALID_PATH)
mask_pf  = pq.ParquetFile(MASK_PATH)

print('=== train.parquet ===')
print(f'  Row groups : {train_pf.num_row_groups:,}  (= sequences)')
print(f'  Columns    : {train_pf.schema_arrow.names}')

print('\n=== valid.parquet ===')
print(f'  Row groups : {valid_pf.num_row_groups:,}')
print(f'  Columns    : {valid_pf.schema_arrow.names}')

print('\n=== valid_mask.parquet ===')
print(f'  Row groups : {mask_pf.num_row_groups:,}')
print(f'  Columns    : {mask_pf.schema_arrow.names}')


## Cell 3 — Load ONE training sequence (row group 0)

In [ ]:
# Reads exactly 20 000 rows — fast
rg0 = train_pf.read_row_group(0).to_pandas()
print(f'Shape: {rg0.shape}')
print(f'seq_ix: {rg0["seq_ix"].iloc[0]}')
print(f'step_in_seq: {rg0["step_in_seq"].min()} → {rg0["step_in_seq"].max()}')
print(f'need_prediction counts:\n{rg0["need_prediction"].value_counts()}')
rg0.head(3)


## Cell 4 — Basic stats on features (one sequence)

In [ ]:
feat = rg0[FEATURE_COLUMNS].astype('float32')
print('Feature summary (first sequence):')
print(feat.describe().T[['mean','std','min','max']].to_string())


## Cell 5 — Target distribution (one training sequence)

In [ ]:
# Only look at prediction rows (step >= 99)
scored = rg0[rg0['need_prediction']]
print(f'Prediction rows: {len(scored):,} out of {len(rg0):,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['t0', 't1']):
    ax.hist(scored[col].clip(-2, 2), bins=80, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(0, color='red', lw=1, ls='--')
    ax.set_title(f'{col} distribution (seq 0)')
    ax.set_xlabel('value (clipped to ±2)')
    ax.set_ylabel('count')
plt.tight_layout()
plt.show()


## Cell 6 — Targets over time (one training sequence)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(scored['step_in_seq'], scored['t0'], lw=0.4, label='t0', alpha=0.8)
ax.plot(scored['step_in_seq'], scored['t1'], lw=0.4, label='t1', alpha=0.8)
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.set_title('t0 and t1 over time (sequence 0)')
ax.set_xlabel('step_in_seq')
ax.set_ylabel('target value')
ax.legend()
plt.tight_layout()
plt.show()


## Cell 7 — Feature groups (what columns exist)

In [ ]:
groups = {}
for col in FEATURE_COLUMNS:
    prefix = col.rstrip('0123456789')
    groups.setdefault(prefix, []).append(col)

print(f'Total features: {len(FEATURE_COLUMNS)}')
for g, cols in groups.items():
    print(f'  {g:10s}: {len(cols):3d} columns  (e.g. {cols[0]}, {cols[-1]})')


## Cell 8 — Sample a few row groups to check variance across sequences

In [ ]:
sample_rgs = [0, 100, 500, 1000, 5000]
rows = []
for i in sample_rgs:
    rg = train_pf.read_row_group(i).to_pandas()
    pred = rg[rg['need_prediction']]
    rows.append({
        'rg': i,
        'seq_ix': rg['seq_ix'].iloc[0],
        't0_mean': round(pred['t0'].mean(), 5),
        't0_std':  round(pred['t0'].std(), 5),
        't1_mean': round(pred['t1'].mean(), 5),
        't1_std':  round(pred['t1'].std(), 5),
    })

pd.DataFrame(rows).set_index('rg')


## Cell 9 — Inspect the valid mask (how many rows scored per sequence?)

In [ ]:
# Valid mask is ~92 MB; still use row-group iteration
scored_counts = []
for i in range(5):
    rg = mask_pf.read_row_group(i).to_pandas()
    scored_counts.append({
        'rg': i,
        'seq_ix': rg['seq_ix'].iloc[0],
        'total_rows': len(rg),
        'is_scored_true': int(rg['is_scored'].sum()),
        'scored_pct': f"{100*rg['is_scored'].mean():.1f}%"
    })

pd.DataFrame(scored_counts).set_index('rg')


## Cell 10 — Inspect baseline solution.py

In [ ]:
baseline_sol = ROOT / 'baseline' / 'solution.py'
print(baseline_sol.read_text())


## Cell 11 — Load baseline ONNX model

In [ ]:
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(ROOT / 'baseline' / 'baseline.onnx'))
    print('ONNX model inputs :')
    for inp in sess.get_inputs():
        print(f'  {inp.name}: {inp.shape}')
    print('ONNX model outputs:')
    for out in sess.get_outputs():
        print(f'  {out.name}: {out.shape}')
    HAS_ONNX = True
except ImportError:
    print('onnxruntime not installed — skipping (pip install onnxruntime)')
    HAS_ONNX = False


## Cell 12 — Run baseline on one valid sequence (requires onnxruntime)

In [ ]:
if not HAS_ONNX:
    print('Skip — onnxruntime not available')
else:
    import pyarrow as pa
    from utils import GlobalAccumulator, validate_sequence

    rg_valid = valid_pf.read_row_group(0)
    seq_ix, need = validate_sequence(rg_valid)

    rg_valid_pd = rg_valid.to_pandas()
    features    = rg_valid_pd[FEATURE_COLUMNS].to_numpy(dtype='float32')   # (20000, 112)
    targets     = rg_valid_pd[list(TARGET_COLUMNS)].to_numpy(dtype='float32')
    mask_col    = rg_valid_pd['is_scored'].to_numpy(dtype=bool) & need

    predictions = np.full((SEQUENCE_LENGTH, 2), np.nan, dtype='float32')

    # Baseline RNN has two hidden states of shape (1, 1, 128) initialized to 0
    hidden_0 = np.zeros((1, 1, 128), dtype=np.float32)
    hidden_1 = np.zeros((1, 1, 128), dtype=np.float32)

    for step in range(SEQUENCE_LENGTH):
        x = features[step].reshape(1, 1, 112)
        pred, hidden_0, hidden_1 = sess.run(
            None,
            {
                'features': x,
                'hidden_0': hidden_0,
                'hidden_1': hidden_1,
            }
        )
        if need[step]:
            predictions[step] = pred[0, 0]

    acc = GlobalAccumulator()
    acc.add(targets, predictions, mask_col)
    result = acc.result()
    print(f'Baseline WP on seq 0: t0={result["t0"]:.4f}, t1={result["t1"]:.4f}, mean={result["weighted_pearson"]:.4f}')


## Cell 13 — Feature–target correlations (one sequence)
Which features have the highest raw correlation with t0 / t1?


In [ ]:
pred_rows = rg0[rg0['need_prediction']].copy()

# ── 1. Top Correlated Features ──────────────────────────────────────────
corr_t0 = pred_rows[FEATURE_COLUMNS].corrwith(pred_rows['t0']).abs().sort_values(ascending=False)
corr_t1 = pred_rows[FEATURE_COLUMNS].corrwith(pred_rows['t1']).abs().sort_values(ascending=False)

print('Top 5 features correlated with t0:')
print(corr_t0.head(5).to_string())
print('\nTop 5 features correlated with t1:')
print(corr_t1.head(5).to_string())

# ── 2. Cross-Instrument Relationship (i0 vs i1 vs a) ───────────────────
i0_cols = [c for c in FEATURE_COLUMNS if c.startswith('i0_')]
i1_cols = [c for c in FEATURE_COLUMNS if c.startswith('i1_')]
a_cols  = [c for c in FEATURE_COLUMNS if c.startswith('a')]

print('\n' + '='*55)
print('CROSS-INSTRUMENT PREDICTABILITY (Mean |r| / Max |r|):')
print(f'  i0 features -> t0: mean={corr_t0[i0_cols].mean():.4f}, max={corr_t0[i0_cols].max():.4f}')
print(f'  i1 features -> t0: mean={corr_t0[i1_cols].mean():.4f}, max={corr_t0[i1_cols].max():.4f}')
print(f'  a  features -> t0: mean={corr_t0[a_cols].mean():.4f}, max={corr_t0[a_cols].max():.4f}')
print(f'  i0 features -> t1: mean={corr_t1[i0_cols].mean():.4f}, max={corr_t1[i0_cols].max():.4f}')
print(f'  i1 features -> t1: mean={corr_t1[i1_cols].mean():.4f}, max={corr_t1[i1_cols].max():.4f}')
print(f'  a  features -> t1: mean={corr_t1[a_cols].mean():.4f}, max={corr_t1[a_cols].max():.4f}')

# ── 3. Target Autocorrelation (How stateful does model need to be?) ───
print('\n' + '='*55)
print('TARGET AUTOCORRELATION:')
lags = [1, 2, 5, 10, 25, 50, 100, 200, 500]
for target_name in ['t0', 't1']:
    ac_vals = [pred_rows[target_name].autocorr(lag=l) for l in lags]
    print(f'  {target_name}: ' + ', '.join(f'lag {l}:{v:.2f}' for l, v in zip(lags[:6], ac_vals[:6])))


## Cell 14 — NaN / Inf check (one sequence)
Verify data quality before training.


In [ ]:
nan_counts = rg0[FEATURE_COLUMNS].isna().sum()
inf_counts  = (~np.isfinite(rg0[FEATURE_COLUMNS].to_numpy(dtype='float32'))).sum(axis=0)

print(f'NaN in features : {nan_counts.sum()} total')
print(f'Inf in features : {inf_counts.sum()} total')
print(f'NaN in t0/t1    : {rg0[["t0","t1"]].isna().sum().sum()}')

if nan_counts.sum() > 0:
    print('\nColumns with NaN:')
    print(nan_counts[nan_counts > 0])


## Cell 15 — Memory usage estimate
One sequence in float32 — how much RAM per batch?


In [ ]:
rows_per_seq = SEQUENCE_LENGTH        # 20 000
n_features   = len(FEATURE_COLUMNS)   # 112
n_targets    = 2
bytes_f32    = 4

feat_mb  = rows_per_seq * n_features * bytes_f32 / 1e6
tgt_mb   = rows_per_seq * n_targets  * bytes_f32 / 1e6
total_mb = feat_mb + tgt_mb

print('One sequence:')
print(f'  Features : {feat_mb:.2f} MB')
print(f'  Targets  : {tgt_mb:.2f} MB')
print(f'  Total    : {total_mb:.2f} MB')

for n in [1, 8, 32, 128]:
    print(f'  {n:4d} seqs : {n * total_mb / 1024:.2f} GB')


---
## ✅ All cells done

Next steps:
- `notebooks/` — look for any existing experiment notebooks
- `src/` — model code lives here
- `baseline/solution.py` — copy this as a starting template for your own model
